# ORAC SEVIRI L2 — exploratory look at R10 vs R11

Goal: load one daytime 15-min slot for both retrievals, eyeball the cloud fields, and look at the diagnostics that distinguish the **clean** (R10) retrieval from the **sequential** (R11) retrieval.

Data root: `/gws/ssde/j25a/cloud_ecv/data_out/seviri/` — month 2026-02.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable when running the notebook in place.
REPO = Path('/gws/pw/j07/nceo_aerosolfire/rsong/project/cloud_cci')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from orac import (
    discover_slots, open_slot, open_paired,
    per_slot_stats, missing_slot_report,
    decode_qcflag, qc_pass_mask, CLDTYPE_NAMES,
    bbox_subset, julian_to_datetime,
)

ROOT = '/gws/ssde/j25a/cloud_ecv/data_out/seviri'

## 1. Pick a daytime slot

SEVIRI is a full-disk geostationary sensor centred at 0°E, so midday UTC is noon over Africa/Europe. We pick 13:12 UTC on 2026-02-25.

In [ ]:
slots = discover_slots(ROOT, start=dt.datetime(2026,2,25), end=dt.datetime(2026,2,26))
target = next(s for s in slots if s.scan_time.hour == 13 and s.scan_time.minute == 12)
print(f'Target slot: {target.scan_time}')
print(f'  Files present: {list(target.files.keys())}')
print(f'  R10 complete: {target.has("R10")}   R11 complete: {target.has("R11")}')

## 2. Load both retrievals together

`open_paired` concatenates R10 and R11 on a new `retrieval` dim.

In [ ]:
core = ['lat','lon','cot','cer','ctp','cth','cwp','cldmask','phase','cldtype','qcflag',
        'niter','costja','costjm','degrees_of_freedom_signal',
        'cot_ap','cot_fg','cer_ap','cer_fg']
both = open_paired(target, variables=core)
print(both)

## 3. Full-disk maps of COT, CER, CTH (R10)

Geostationary projection means we can plot on-disk `(along_track, across_track)` indices directly. Off-disk pixels are NaN already.

In [ ]:
r10 = both.sel(retrieval='R10').squeeze(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, var, title, kw in [
    (axes[0], 'cot', 'Cloud optical thickness',    dict(vmin=0, vmax=60, cmap='viridis')),
    (axes[1], 'cer', 'Cloud effective radius (µm)', dict(vmin=2, vmax=30, cmap='magma')),
    (axes[2], 'cth', 'Cloud top height (km)',      dict(vmin=0, vmax=15, cmap='cividis')),
]:
    im = ax.imshow(np.asarray(r10[var].values), origin='upper', **kw)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle(f'R10, {target.scan_time}')
plt.tight_layout()

## 4. R11 − R10 difference maps

If the sequential prior works, these should be small but non-zero, and spatially coherent.

In [ ]:
r11 = both.sel(retrieval='R11').squeeze(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, var, title, scale in [
    (axes[0], 'cot', 'Δ COT', 5),
    (axes[1], 'cer', 'Δ CER (µm)', 3),
    (axes[2], 'cth', 'Δ CTH (km)', 2),
]:
    diff = np.asarray(r11[var].values) - np.asarray(r10[var].values)
    im = ax.imshow(diff, origin='upper', vmin=-scale, vmax=scale, cmap='RdBu_r')
    ax.set_title(f'R11 − R10: {title}')
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()

## 5. Retrieval diagnostics: iterations and cost

The hope is R11 converges in fewer iterations because the prior is closer to truth.

In [ ]:
lat = np.asarray(r10['lat'].values)
on_disk = np.isfinite(lat)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, var, rng in [
    (axes[0], 'niter',   (0, 50)),
    (axes[1], 'costja',  (0, 5)),
    (axes[2], 'costjm',  (0, 30)),
]:
    for r, colour in [('R10', 'tab:blue'), ('R11', 'tab:orange')]:
        arr = np.asarray(both.sel(retrieval=r).squeeze(drop=True)[var].values)
        vals = arr[on_disk]
        vals = vals[np.isfinite(vals)]
        ax.hist(vals, bins=80, range=rng, alpha=0.5, label=r, color=colour, density=True)
    ax.set_title(var)
    ax.set_xlabel(var)
    ax.legend()
plt.tight_layout()

## 6. QC breakdown (R10 only)

In [ ]:
qc = decode_qcflag(r10['qcflag'])
# Count only on-disk pixels.
on_disk_da = np.isfinite(r10['lat'])
counts = {name: int((qc[name] & on_disk_da).sum().values) for name in qc.data_vars}
n_ondisk = int(on_disk_da.sum().values)
qc_df = (pd.DataFrame({'n': counts})
         .assign(fraction=lambda d: d['n'] / n_ondisk)
         .sort_values('n', ascending=False))
qc_df

## 7. Cloud-type histogram (Pavolonis)

In [ ]:
ct = np.asarray(r10['cldtype'].squeeze(drop=True).values)
ct_on = ct[on_disk]
ct_on = ct_on[np.isfinite(ct_on)]
vals, counts = np.unique(ct_on.astype(int), return_counts=True)
pd.DataFrame({
    'code': vals,
    'name': [CLDTYPE_NAMES[v] if 0 <= v < len(CLDTYPE_NAMES) else 'unknown' for v in vals],
    'n':    counts,
    'fraction': counts / on_disk.sum(),
}).sort_values('n', ascending=False)

## 8. R10 vs R11 per-slot stats for one whole day

In [ ]:
daytime = [s for s in slots if 6 <= s.scan_time.hour <= 18]
rows = []
for s in daytime[::4]:  # every hour
    for r in ('R10', 'R11'):
        if not s.has(r):
            continue
        ds = open_slot(s, r, variables=['lat','lon','cot','cer','cth','qcflag','cldmask',
                                         'niter','costja','costjm','degrees_of_freedom_signal'])
        stats = per_slot_stats(ds, variables=['cot','cer','cth'], qc_rules='default')
        stats['scan_time'] = s.scan_time
        stats['retrieval'] = r
        rows.append(stats)
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, col in zip(axes, ['niter_median', 'costja_median', 'cot_median']):
    for r, colour in [('R10', 'tab:blue'), ('R11', 'tab:orange')]:
        sub = df[df.retrieval == r]
        ax.plot(sub.scan_time, sub[col], 'o-', label=r, color=colour)
    ax.set_title(col); ax.legend()
    ax.grid(alpha=0.3)
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
df

## 9. Missing-slot report for the whole month

In [ ]:
rep = missing_slot_report(ROOT, 2026, 2)
print(f'Expected slots:        {len(rep)}')
print(f'R10.primary present:   {int(rep.R10_primary.sum())}')
print(f'R11.primary present:   {int(rep.R11_primary.sum())}')

miss_r10 = rep[~rep.R10_primary][['expected_slot']].head(20)
print('\nFirst 20 slots with R10.primary missing:')
print(miss_r10)

## 10. Europe bbox close-up

In [ ]:
eu = bbox_subset(r10, lon=(-15, 35), lat=(30, 65))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(np.asarray(eu.cot.values), origin='upper', vmin=0, vmax=40, cmap='viridis')
axes[0].set_title('Europe: COT (R10)')
axes[1].imshow(np.asarray(eu.cth.values), origin='upper', vmin=0, vmax=12, cmap='cividis')
axes[1].set_title('Europe: CTH (R10)')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()